# Rule-Based Cell Phenotyping

This notebook assigns simplified phenotype labels to the processed single-cell table. Phenotyping is the step that converts marker measurements into interpretable cell categories, such as plasma-cell-like, T-cell-like, myeloid-like, stromal-like, or unknown.

The input table contains one row per segmented cell object. Each row includes transformed marker values, object area, spatial coordinates, and image identifiers. The phenotyping step adds two new interpretation columns:

```text
category
phenotype
```

The `category` column is inferred from the image name, such as `MGUS`, `UB`, or `B`. The `phenotype` column is assigned using transparent marker rules.

Important methodological boundary:

- The template paper used expert-guided manual annotation, an optimized XGBoost classifier, and FlowSOM review for unresolved cells.
- This notebook does not reproduce the full machine-learning annotation workflow from the template paper.
- This notebook implements an interpretable rule-based approximation for the representative subset.
- Phenotype names use the suffix `_like` to indicate that labels are approximate marker-pattern labels, not definitive expert-reviewed identities.

Reason for using this simplified step:

- The rule-based method is easier to inspect and understand.
- Each assigned label can be traced back to specific marker conditions.
- The method provides a practical first annotation layer before more advanced annotation methods are attempted.
- The output supports composition summaries and preliminary spatial analysis while preserving clear interpretation boundaries.

Main input:

```text
results/11_processed_single_cell_features.csv
```

Main outputs:

```text
results/13_cells_with_phenotypes.csv
results/14_phenotype_composition_by_image.csv
results/15_phenotype_composition_by_category.csv
results/16_rule_based_phenotyping_thresholds.csv
figures/04_phenotype_counts_by_image.png
figures/05_phenotype_composition_by_category.png
```


## Step 0: Configure Workflow Paths

This setup cell defines workflow paths and confirms that required input files and scripts are present.

The phenotyping step starts from the processed single-cell table rather than the raw Steinbock export. This is important because marker columns have already been prepared according to the template-paper feature-processing description:

```text
area filtering
-> 99th percentile censoring
-> arcsinh transformation with cofactor 1
```

Using the processed table keeps phenotyping consistent with the transformed marker scale used for downstream analysis. The raw exported table remains unchanged and available for reference.


In [ ]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

RESULTS_DIR = WORKFLOW_DIR / 'results'
FIGURES_DIR = WORKFLOW_DIR / 'figures'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

INPUT = RESULTS_DIR / '11_processed_single_cell_features.csv'
OUTPUT = RESULTS_DIR / '13_cells_with_phenotypes.csv'
COMPOSITION_IMAGE = RESULTS_DIR / '14_phenotype_composition_by_image.csv'
COMPOSITION_CATEGORY = RESULTS_DIR / '15_phenotype_composition_by_category.csv'
THRESHOLDS = RESULTS_DIR / '16_rule_based_phenotyping_thresholds.csv'

print('Workflow directory:', WORKFLOW_DIR)
print('Processed single-cell table exists:', INPUT.exists())
print('Phenotyping script exists:', (SCRIPTS_DIR / '10_rule_based_phenotyping.py').exists())


## Step 1: Understand the Rule-Based Phenotyping Strategy

Rule-based phenotyping uses marker combinations to assign approximate cell identities. A marker is first converted into a positive/negative decision. The script calculates this marker positivity threshold from the processed data itself. By default, a marker is considered positive when its transformed value is at or above the 80th percentile for that marker.

This percentile threshold is a practical starting point. It identifies cells with relatively high expression of a marker within this dataset. It is not the same as manual gating by an expert pathologist or cytometrist, but it makes the annotation logic explicit and reproducible.

Marker interpretation used in this first-pass rule set:

```text
CD138, CD38, IRF4: plasma-cell-associated markers
CD3: T-cell-associated marker
CD4: CD4 T-cell-associated marker
CD8: CD8 T-cell-associated marker
CD45: broad immune/leukocyte marker
CD68, CD11b, CD11c, HLA-DR: myeloid/macrophage/dendritic-associated markers
MPO: neutrophil/granulocyte-associated marker
CD34: progenitor/endothelial-associated marker in this simplified rule set
Perilipin: adipocyte-associated marker
CathepsinK: osteoclast-associated marker
RUNX2, CollagenTypeI, Vimentin: osteoblast/stromal-associated markers
```

Rule hierarchy used by the script:

```text
Plasma_cell_like: CD138 positive and either CD38 or IRF4 positive
CD4_T_cell_like: CD3 positive, CD4 positive, CD8 not positive
CD8_T_cell_like: CD3 positive, CD8 positive, CD4 not positive
T_cell_like: CD3 positive but not cleanly CD4 or CD8
Neutrophil_like: MPO positive and CD11b positive
Macrophage_monocyte_like: CD68 positive and either CD11b or CD45 positive
Dendritic_myeloid_like: HLA-DR positive and CD11c positive
Myeloid_cell_like: CD45 positive with CD68, CD11b, or HLA-DR positivity
CD34_positive_cell_like: CD34 positive
Adipocyte_like: Perilipin positive
Osteoclast_like: CathepsinK positive
Osteoblast_stromal_like: RUNX2 positive and CollagenTypeI positive
Stromal_like: Vimentin positive and CollagenTypeI positive
Unknown: no rule matched
```

Why rule order matters:

- A cell can express more than one marker above threshold.
- More specific rules should be evaluated before broader rules.
- For example, a plasma-cell-like label is assigned before broader immune or stromal rules.
- A broad label such as `Myeloid_cell_like` is used only after more specific myeloid-like rules have been evaluated.

Meaning of the `Unknown` class:

- `Unknown` does not mean that the cell is invalid.
- `Unknown` means that the current simplified marker rules did not assign a confident approximate phenotype.
- A large `Unknown` group is expected in conservative first-pass phenotyping.
- Reducing `Unknown` usually requires improved marker rules, reference annotation, clustering, or supervised classification.


## Step 2: Run Rule-Based Phenotyping

The script performs the complete rule-based annotation workflow.

Processing sequence:

```text
1. Read the processed single-cell table.
2. Calculate marker positivity thresholds from selected lineage markers.
3. Evaluate marker rules for every cell.
4. Assign one phenotype label per cell.
5. Infer disease/sample category from the image name.
6. Save the phenotyped single-cell table.
7. Summarize phenotype composition by image and by category.
8. Save figures for visual review.
```

Input table:

```text
results/11_processed_single_cell_features.csv
```

Output table with one row per cell:

```text
results/13_cells_with_phenotypes.csv
```

Additional summary outputs:

```text
results/14_phenotype_composition_by_image.csv
results/15_phenotype_composition_by_category.csv
results/16_rule_based_phenotyping_thresholds.csv
```

Figure outputs:

```text
figures/04_phenotype_counts_by_image.png
figures/05_phenotype_composition_by_category.png
```


In [ ]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '10_rule_based_phenotyping.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--input', 'results/11_processed_single_cell_features.csv',
    '--output-cells', 'results/13_cells_with_phenotypes.csv',
    '--composition-image', 'results/14_phenotype_composition_by_image.csv',
    '--composition-category', 'results/15_phenotype_composition_by_category.csv',
    '--thresholds-output', 'results/16_rule_based_phenotyping_thresholds.csv',
    '--counts-figure', 'figures/04_phenotype_counts_by_image.png',
    '--category-figure', 'figures/05_phenotype_composition_by_category.png',
    '--positive-percentile', '80',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


## Step 3: Review Marker Positivity Thresholds

The threshold table records the cutoff used to decide whether each lineage marker is positive. This table is essential for transparency because it defines how marker values were converted into phenotype rules.

Columns in the threshold table:

```text
marker: marker name
positive_percentile: percentile used to calculate positivity threshold
positive_threshold: transformed marker value at that percentile
```

How to interpret thresholds:

- A cell is considered positive for a marker when its transformed marker value is greater than or equal to the marker threshold.
- A high threshold means that only cells with stronger relative marker expression are considered positive.
- A low threshold means that the transformed marker distribution has a lower range in this subset.
- Thresholds are calculated independently for each marker.

Interpretation caution:

- Percentile thresholds are data-derived and simple.
- They are not equivalent to expert biological gating.
- They provide a reproducible first-pass definition of marker positivity for this representative subset.


In [ ]:
with THRESHOLDS.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    threshold_rows = list(reader)

for row in threshold_rows:
    print(row)


## Step 4: Review Phenotype Composition by Image

The image-level composition table summarizes rule-based phenotypes separately for each ROI image. Each row contains one image and one phenotype class.

Columns in the image-level table:

```text
Image: source ROI image
phenotype: rule-based phenotype label
cell_count: number of cells assigned to that phenotype in the image
fraction: phenotype fraction among all cells in that image
```

Why this table is useful:

- It confirms that every ROI has phenotyped cells.
- It shows whether certain phenotypes are concentrated in specific images.
- It helps identify possible image-specific segmentation or staining behavior.
- It prepares the data for later spatial and category-level interpretation.

Interpretation caution:

- Differences between images can reflect biology, tissue sampling, staining variation, segmentation behavior, or rule sensitivity.
- This table is exploratory until phenotype labels are validated against expert annotation or reference labels.


In [ ]:
with COMPOSITION_IMAGE.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    image_rows = list(reader)

for row in image_rows[:30]:
    print(row)


## Step 5: Review Phenotype Composition by Category

The category-level composition table groups cells by disease/sample category inferred from image names. In this representative subset, categories include `MGUS`, `UB`, and `B`.

Columns in the category-level table:

```text
category: category inferred from image name
phenotype: rule-based phenotype label
cell_count: number of cells assigned to the phenotype in that category
fraction: phenotype fraction among all cells in that category
```

Purpose of this table:

- Summarize broad phenotype patterns across sample categories.
- Provide a compact overview for comparison across `MGUS`, `UB`, and `B` images.
- Support early checks before spatial analysis or biological interpretation.

Interpretation caution:

- The subset is small and not designed for final statistical inference.
- Category labels are inferred from filenames.
- Phenotype labels are rule-based approximations.
- This output should be treated as an exploratory summary, not a final disease-associated biological result.


In [ ]:
with COMPOSITION_CATEGORY.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    category_rows = list(reader)

for row in category_rows:
    print(row)


## Step 6: Review Generated Figures

The generated figures provide visual checks of the rule-based annotation output.

Figure interpretation:

- `04_phenotype_counts_by_image.png`: stacked phenotype counts for each ROI image. This figure highlights image-level differences in absolute cell numbers and phenotype assignments.
- `05_phenotype_composition_by_category.png`: stacked phenotype fractions by category. This figure highlights broad category-level composition patterns.

Expected observations:

- All eight images should appear in the image-level figure.
- Category-level fractions should sum to approximately 1 within each category.
- A visible `Unknown` fraction is expected because the rule set is conservative.

Interpretation caution:

- These figures summarize rule-based labels only.
- They should not be interpreted as final reproduction of the template paper phenotyping.
- Strong biological claims require validated annotation, larger sample context, and appropriate statistical testing.


In [ ]:
expected_figures = [
    FIGURES_DIR / '04_phenotype_counts_by_image.png',
    FIGURES_DIR / '05_phenotype_composition_by_category.png',
]

for figure in expected_figures:
    print(figure.name, 'exists:', figure.exists(), 'size_bytes:', figure.stat().st_size if figure.exists() else 0)
